# Imbalanced-Aware Categorical (Multi-Label) Toxicity Classification

Applies the friend's pipeline to **multi-label** detection: predicting all 6 toxicity categories simultaneously.

**Key adaptation from binary**: SMOTE variants do not support multi-label targets.  
Oversampling comparison uses `RandomOverSampler` (which does) vs no sampling.  
All other design principles are preserved:

- All parameters in `CONFIG`
- `ImbPipeline` keeps oversampling inside each CV fold
- Train + Val combined — 5-fold CV replaces a fixed validation split
- OOF probabilities used for per-label threshold tuning

**Stages**:
1. Oversampling comparison — no sampling vs RandomOverSampler via 5-fold CV
2. Model comparison — MultiOutput(LogisticRegressionCV), MultiOutput(ComplementNB), MultiOutput(LinearSVC via Optuna TPE)
3. Final evaluation — per-label + macro metrics on test set
4. Interpretability — per-label top features, error analysis

In [ ]:
import sys, warnings, time
sys.path.insert(0, '../models')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, roc_auc_score, classification_report, precision_recall_fscore_support

from baseline import _feature_union, ALL_STOPWORDS, RANDOM_STATE
from preprocessing import preprocess_df

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

CONFIG = {
    'paths': {
        'train':       '../data/train_cleaned.parquet',
        'test':        '../data/test_cleaned.parquet',
        'test_labels': '../data/test_labels_cleaned.parquet',
    },
    'cv': {
        'n_splits': 5,
        'scoring':  'f1_macro',
    },
    'optuna': {
        'n_trials': 30,
    },
    'random_state': RANDOM_STATE,
}

## 1. Load & Prepare Data

In [ ]:
train_raw   = pd.read_parquet(CONFIG['paths']['train'])
test_raw    = pd.read_parquet(CONFIG['paths']['test'])
test_labels = pd.read_parquet(CONFIG['paths']['test_labels'])

# Binary collapse (needed for stratified CV proxy and two-stage demo)
train_raw['is_toxic'] = (train_raw[LABELS].sum(axis=1) > 0).astype(int)

# Test set: filter withheld rows, collapse labels, safe merge for comment_text
# (test_labels_cleaned already has comment_text — drop before merge to avoid _x/_y suffixes)
evaluable = test_labels[test_labels['toxic'] != -1].copy()
evaluable['is_toxic'] = (evaluable[LABELS].sum(axis=1) > 0).astype(int)
eval_df = (
    evaluable
    .drop(columns=['comment_text'], errors='ignore')
    .merge(test_raw[['id', 'comment_text']], on='id', how='left')
    .reset_index(drop=True)
)

y_train  = train_raw[LABELS].values   # (n_train, 6)
y_test   = eval_df[LABELS].values     # (n_test,  6)

print(f'Train : {len(train_raw):,} rows')
print(f'Test  : {len(eval_df):,} rows')
print('\nPositive rate per label (train):')
for label in LABELS:
    print(f'  {label:<20} {train_raw[label].mean()*100:.2f}%')

## 2. Preprocess & Extract Features

Same trade-off as the binary notebook: TF-IDF vocabulary is fit on all training data before CV  
(minor leakage), but oversampling stays strictly inside each fold.

In [ ]:
print('Preprocessing text...')
train_text = preprocess_df(train_raw['comment_text'])
test_text  = preprocess_df(eval_df['comment_text'])

print('Extracting features...')
_word_tfidf = TfidfVectorizer(
    sublinear_tf=True, max_features=100_000, ngram_range=(1, 2),
    min_df=3, max_df=0.95, norm='l2', analyzer='word', stop_words=ALL_STOPWORDS,
)
X_train_raw = _word_tfidf.fit_transform(train_text)
X_test_raw  = _word_tfidf.transform(test_text)

scaler  = MaxAbsScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

# Separate unscaled features for ComplementNB (requires non-negative inputs)
nb_tfidf = TfidfVectorizer(
    sublinear_tf=True, max_features=100_000, ngram_range=(1, 2),
    min_df=3, max_df=0.95, norm='l2', analyzer='word', stop_words=ALL_STOPWORDS,
)
X_train_nb = nb_tfidf.fit_transform(train_text)
X_test_nb  = nb_tfidf.transform(test_text)

# StratifiedKFold on binary is_toxic as a proxy — StratifiedKFold doesn't support
# multi-label targets directly, but stratifying on the binary collapse preserves
# the overall toxic/non-toxic ratio in every fold.
y_binary = train_raw['is_toxic'].values
rs  = CONFIG['random_state']
skf = StratifiedKFold(n_splits=CONFIG['cv']['n_splits'], shuffle=True, random_state=rs)
kf  = list(skf.split(X_train, y_binary))   # pre-computed indices, reused by all CV cells

print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')

## 3. Oversampling — Multi-Label Constraint

SMOTE and variants require a single-output target and do **not** support multi-label.  
`RandomOverSampler` nominally supports multi-label but imbalanced-learn's pipeline validation  
rejects multi-output arrays at the current installed version.

The binary notebook already showed `class_weight='balanced'` outperformed every sampler.  
We apply the same conclusion here: use per-label `class_weight='balanced'` and skip oversampling.

In [ ]:
# No oversampling — class_weight='balanced' handles imbalance per label.
best_sampler_name = 'None (class_weight)'
best_sampler      = None
print(f'Sampler: {best_sampler_name}')

## 4. Model Comparison (with best sampler)

In [ ]:
# --- MultiOutput(LogisticRegressionCV): auto-selects C per label ---
mo_lr_cv = MultiOutputClassifier(
    LogisticRegressionCV(
        Cs=10, cv=3, scoring='f1_macro',
        class_weight='balanced', solver='liblinear',
        max_iter=1000, random_state=rs,
    ),
    n_jobs=-1,
)
pipe_lr = (
    ImbPipeline([('sampler', best_sampler), ('clf', mo_lr_cv)])
    if best_sampler is not None
    else Pipeline([('clf', mo_lr_cv)])
)
scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=kf,
                             scoring=CONFIG['cv']['scoring'], n_jobs=1)
print(f'MultiOutput(LR-CV)    macro F1: {scores_lr.mean():.4f} ± {scores_lr.std():.4f}')

In [ ]:
# --- MultiOutput(ComplementNB): word TF-IDF only ---
mo_nb = MultiOutputClassifier(ComplementNB(), n_jobs=-1)
pipe_nb = (
    ImbPipeline([('sampler', best_sampler), ('clf', mo_nb)])
    if best_sampler is not None
    else Pipeline([('clf', mo_nb)])
)
scores_nb = cross_val_score(pipe_nb, X_train_nb, y_train, cv=kf,
                             scoring=CONFIG['cv']['scoring'], n_jobs=1)
print(f'MultiOutput(NB)       macro F1: {scores_nb.mean():.4f} ± {scores_nb.std():.4f}')

In [ ]:
# --- MultiOutput(LinearSVC): Optuna TPE searches C ---
def svc_objective(trial):
    C = trial.suggest_float('C', 1e-3, 10.0, log=True)
    clf = MultiOutputClassifier(
        LinearSVC(C=C, class_weight='balanced', max_iter=5000, random_state=rs),
        n_jobs=-1,
    )
    pipe = (
        ImbPipeline([('sampler', best_sampler), ('clf', clf)])
        if best_sampler is not None
        else Pipeline([('clf', clf)])
    )
    return cross_val_score(pipe, X_train, y_train, cv=kf,
                           scoring=CONFIG['cv']['scoring'], n_jobs=1).mean()

study_svc = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=rs),
)
study_svc.optimize(svc_objective, n_trials=CONFIG['optuna']['n_trials'], show_progress_bar=True)

best_C_svc = study_svc.best_params['C']
scores_svc = study_svc.best_value
print(f'MultiOutput(SVC)      macro F1: {scores_svc:.4f}  best C={best_C_svc:.4f}')

In [ ]:
model_cv_scores = {
    'MultiOutput(LR-CV)': scores_lr.mean(),
    'MultiOutput(NB)':    scores_nb.mean(),
    'MultiOutput(SVC)':   scores_svc,
}
comparison = pd.DataFrame([
    {'Model': k, 'CV Macro F1': round(v, 4)}
    for k, v in model_cv_scores.items()
]).sort_values('CV Macro F1', ascending=False)
print(comparison.to_string(index=False))

best_model_name = comparison.iloc[0]['Model']
print(f'\n→ Best model: {best_model_name}')

## 5. Final Evaluation on Test Set

Best model fit on full training set.  
Per-label thresholds tuned on OOF predicted probabilities.

In [ ]:
if best_model_name == 'MultiOutput(LR-CV)':
    final_clf = MultiOutputClassifier(
        LogisticRegressionCV(Cs=10, cv=3, scoring='f1_macro',
                             class_weight='balanced', solver='liblinear',
                             max_iter=1000, random_state=rs),
        n_jobs=-1,
    )
    X_fit, X_eval_final = X_train, X_test
elif best_model_name == 'MultiOutput(NB)':
    final_clf = MultiOutputClassifier(ComplementNB(), n_jobs=-1)
    X_fit, X_eval_final = X_train_nb, X_test_nb
else:  # MultiOutput(SVC)
    final_clf = MultiOutputClassifier(
        LinearSVC(C=best_C_svc, class_weight='balanced', max_iter=5000, random_state=rs),
        n_jobs=-1,
    )
    X_fit, X_eval_final = X_train, X_test

final_pipe = (
    ImbPipeline([('sampler', best_sampler), ('clf', final_clf)])
    if best_sampler is not None
    else Pipeline([('clf', final_clf)])
)

# OOF probabilities via manual fold loop.
# cross_val_predict does not handle MultiOutputClassifier.predict_proba correctly
# (returns a list of arrays per label, not a single array).
print('Computing OOF probabilities for per-label threshold tuning...')
oof_proba = np.zeros((len(y_train), len(LABELS)))
for fold, (train_idx, val_idx) in enumerate(kf):
    pipe_fold = clone(final_pipe)
    pipe_fold.fit(X_fit[train_idx], y_train[train_idx])
    proba_list = pipe_fold.predict_proba(X_fit[val_idx])
    for j, p in enumerate(proba_list):
        oof_proba[val_idx, j] = p[:, 1]
    print(f'  Fold {fold+1} done.')

# Per-label threshold tuning
thresholds = np.full(len(LABELS), 0.5)
for i, label in enumerate(LABELS):
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.1, 0.9, 0.02):
        f = f1_score(y_train[:, i], (oof_proba[:, i] >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    thresholds[i] = best_t
print('\nPer-label thresholds:', {l: round(float(t), 2) for l, t in zip(LABELS, thresholds)})

# Fit on full training set, predict on test
final_pipe.fit(X_fit, y_train)
test_proba_list = final_pipe.predict_proba(X_eval_final)
test_proba = np.column_stack([p[:, 1] for p in test_proba_list])   # (n_test, 6)
test_pred  = (test_proba >= thresholds).astype(int)

In [ ]:
# Per-label metrics
rows = []
for i, label in enumerate(LABELS):
    p, r, f, _ = precision_recall_fscore_support(
        y_test[:, i], test_pred[:, i], average='binary', zero_division=0
    )
    auc = roc_auc_score(y_test[:, i], test_proba[:, i]) if y_test[:, i].sum() > 0 else float('nan')
    rows.append({'Label': label, 'Precision': round(p,4), 'Recall': round(r,4),
                 'F1': round(f,4), 'AUC': round(auc,4), 'Threshold': round(thresholds[i],2)})

metrics_df = pd.DataFrame(rows).set_index('Label')
macro_f1  = f1_score(y_test, test_pred, average='macro', zero_division=0)
micro_f1  = f1_score(y_test, test_pred, average='micro', zero_division=0)
macro_auc = np.nanmean([r['AUC'] for r in rows])

print(f'\n{"="*65}')
print(f'  {best_model_name}  [sampler: {best_sampler_name}]')
print(f'{"="*65}')
print(metrics_df.to_string())
print(f'{"─"*65}')
print(f'  Macro F1 : {macro_f1:.4f}   Micro F1 : {micro_f1:.4f}   Macro AUC : {macro_auc:.4f}')

In [ ]:
# Per-label F1 bar chart
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(metrics_df.index, metrics_df['F1'], color=sns.color_palette('muted', len(LABELS)))
ax.axhline(macro_f1, color='red', ls='--', lw=1.5, label=f'macro F1 = {macro_f1:.4f}')
for bar, f1 in zip(bars, metrics_df['F1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{f1:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('F1')
ax.set_title(f'{best_model_name} — Per-Label F1')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Interpretability

Per-label top features from a LogisticRegression fit with the best sampler.

In [ ]:
from sklearn.multioutput import MultiOutputClassifier as MOC

lr_interp = MOC(
    LogisticRegression(C=1.0, class_weight='balanced', solver='liblinear',
                       max_iter=1000, random_state=rs),
    n_jobs=-1,
)
interp_pipe = (
    ImbPipeline([('sampler', best_sampler), ('clf', lr_interp)])
    if best_sampler is not None
    else Pipeline([('clf', lr_interp)])
)
interp_pipe.fit(X_train, y_train)

feat_names = np.array(_word_tfidf.get_feature_names_out())

estimators = interp_pipe.named_steps['clf'].estimators_
top_n = 15

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, estimator, label in zip(axes.ravel(), estimators, LABELS):
    coef = estimator.coef_[0]
    idx  = np.argsort(coef)[-top_n:][::-1]
    ax.barh(feat_names[idx][::-1], coef[idx][::-1], color='tomato')
    ax.set_title(f'Top features — {label}', fontsize=10)
    ax.set_xlabel('LR coefficient')

plt.suptitle('Per-Label Top Toxic Features (Logistic Regression)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Error analysis: per-label false negative rate and false positive rate
error_rows = []
for i, label in enumerate(LABELS):
    tn = int(((y_test[:, i] == 0) & (test_pred[:, i] == 0)).sum())
    fp = int(((y_test[:, i] == 0) & (test_pred[:, i] == 1)).sum())
    fn = int(((y_test[:, i] == 1) & (test_pred[:, i] == 0)).sum())
    tp = int(((y_test[:, i] == 1) & (test_pred[:, i] == 1)).sum())
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    error_rows.append({'Label': label, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                       'FPR': round(fpr, 3), 'FNR (miss rate)': round(fnr, 3)})

error_df = pd.DataFrame(error_rows).set_index('Label')
print('Per-label error breakdown:')
print(error_df.to_string())

# Show hardest false negatives for the 'toxic' label
toxic_idx = LABELS.index('toxic')
mask_fn = (y_test[:, toxic_idx] == 1) & (test_pred[:, toxic_idx] == 0)
fn_df = eval_df[mask_fn][['comment_text']].copy()
fn_df['proba'] = test_proba[mask_fn, toxic_idx].round(3)
print(f'\nTop 5 false negatives (toxic label, lowest proba):')
print(fn_df.nsmallest(5, 'proba')[['comment_text', 'proba']].to_string(index=False))

## 7. Two-Stage Inference Demo

**Pipeline:**
```
Input text → [Binary gate] → Non-toxic → done
                           ↓ Toxic
                    [Multi-label model] → {obscene, insult, threat, …}
```

Stage 1 re-fits a fresh `LogisticRegression` binary model (same TF-IDF features, same CV splits).  
Stage 2 passes only the flagged rows to the already-fitted `final_pipe` for label reasoning.

> **Note:** Once this notebook runs error-free end-to-end, replace Stage 1 with saved predictions
> from `imbalanced_binary_classification.ipynb` (see the commented block at the bottom of this section).

In [ ]:
# ── Stage 1: Load binary predictions from imbalanced_binary_classification.ipynb
print('Stage 1 — Binary gate (loading saved predictions)')

from pathlib import Path
_bin = pd.read_parquet(Path('../data/binary_predictions.parquet'))
bin_test_proba = _bin['binary_proba'].values
best_bin_t     = float(_bin['binary_threshold'].iloc[0])
bin_test_pred  = _bin['binary_pred'].values

y_test_binary = eval_df['is_toxic'].values
n_flagged = int(bin_test_pred.sum())
print(f'  Threshold        : {best_bin_t:.2f}')
print(f'  Flagged as toxic : {n_flagged:,} / {len(bin_test_pred):,} ({n_flagged/len(bin_test_pred)*100:.1f}%)')
print(f'  Binary F1 on test: {f1_score(y_test_binary, bin_test_pred, zero_division=0):.4f}')

In [ ]:
# ── Stage 2: Multi-label reasoning on flagged rows ────────────────────────────
print('Stage 2 — Multi-label classifier on flagged rows only\n')

flagged_mask   = bin_test_pred == 1
X_test_flagged = X_eval_final[flagged_mask]

multi_proba_flagged = np.column_stack([
    p[:, 1] for p in final_pipe.predict_proba(X_test_flagged)
])
multi_pred_flagged = (multi_proba_flagged >= thresholds).astype(int)

# Build combined results: non-flagged rows get all-zero label vectors
results_df = eval_df[['id', 'comment_text', 'is_toxic']].copy()
results_df['binary_proba'] = bin_test_proba.round(3)
results_df['binary_pred']  = bin_test_pred
for j, label in enumerate(LABELS):
    results_df[label] = 0
results_df.loc[flagged_mask, LABELS] = multi_pred_flagged

# ── Evaluation ────────────────────────────────────────────────────────────────
twostage_macro_f1 = f1_score(y_test, results_df[LABELS].values, average='macro', zero_division=0)
direct_macro_f1   = f1_score(y_test, test_pred, average='macro', zero_division=0)

print(f'Macro F1 — two-stage pipeline : {twostage_macro_f1:.4f}')
print(f'Macro F1 — direct multi-label : {direct_macro_f1:.4f}')
print('(Two-stage: binary gate filters first; direct: multi-label runs on every row)')

# ── Label distribution among flagged rows ─────────────────────────────────────
print(f'\nLabel breakdown for {n_flagged:,} flagged (toxic) rows:')
for label in LABELS:
    cnt = int(results_df.loc[flagged_mask, label].sum())
    pct = cnt / n_flagged * 100
    print(f'  {label:<20} {cnt:>5,}  ({pct:.1f}%)')

# ── Sample output ─────────────────────────────────────────────────────────────
print('\n5 sample flagged comments with label reasoning:')
pd.set_option('display.max_colwidth', 80)
display(
    results_df[flagged_mask][['comment_text', 'binary_proba'] + LABELS]
    .head(5)
)